# Feed Generation Pipeline Visualization

`agents/feed_generation/pipeline.py` 는 LangGraph 8-노드 직렬 파이프라인이다.

본 노트북은 세 갈래로 검증한다.

1. **LangGraph 그래프 토폴로지** — 컴파일된 그래프에서 Mermaid 자동 추출
2. **architecture.mmd 다이어그램** — 설계 문서 다이어그램 렌더링
3. **Fake 포트로 파이프라인 실행** — 정상 흐름 + 4가지 실패 시나리오 직접 확인

파이프라인 요약:
```
validate → assemble_image_prompt → img2img(retry=3) → s3_upload(retry=3)
  → assemble_caption_ctx → llm_caption(retry=3) → validate_caption → builder
```
VLM 없음. 캐릭터 기준 이미지(S3 URL) → Img2Img → S3 업로드 → Mi:dm 캡션 → 검증.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from agents.feed_generation import pipeline
from agents.feed_generation.exceptions import (
    CaptionGenerationError,
    CaptionValidationError,
    ImageGenerationError,
    InputValidationError,
    S3UploadError,
)
from agents.feed_generation.graph import build_graph
from agents.feed_generation.protocols import Ports
from agents.feed_generation.schemas import (
    CharacterRef,
    FeedGenerationInput,
    GeneratedFeed,
    QuestRef,
)

print(pipeline.run.__module__, "·", pipeline.run.__qualname__)

## 1. LangGraph 그래프 토폴로지

컴파일된 그래프에서 Mermaid 소스를 자동 추출한다.
`graph.py` 의 `add_edge` 선언과 일치하는지 확인.

In [ ]:
graph = build_graph()
mermaid_graph = graph.get_graph().draw_mermaid()
print(mermaid_graph)

In [ ]:
from IPython.display import HTML, display


def render_mermaid(src: str) -> None:
    html = f"""
<div class="mermaid">
{src}
</div>
<script type="module">
  import mermaid from 'https://cdn.jsdelivr.net/npm/mermaid@11/dist/mermaid.esm.min.mjs';
  mermaid.initialize({{ startOnLoad: true }});
</script>
"""
    display(HTML(html))


render_mermaid(mermaid_graph)

## 2. architecture.mmd 소스

`docs/features/feed_generation/architecture.mmd` 를 직접 읽어 출력한다.
https://mermaid.live 에 붙여넣으면 렌더링됨.

In [ ]:
mmd_path = ROOT / "docs" / "features" / "feed_generation" / "architecture.mmd"
arch_src = mmd_path.read_text(encoding="utf-8")
print(arch_src)

In [ ]:
render_mermaid(arch_src)

## 3. I/O 스키마 확인

Pydantic 모델 필드를 직접 출력 — `schemas.py` 변경 시 빠른 검증용.

In [ ]:
def show(model):
    print(f"=== {model.__name__} ===")
    for name, field in model.model_fields.items():
        print(f"  - {name}: {field.annotation}")
    print()


for m in (QuestRef, CharacterRef, FeedGenerationInput, GeneratedFeed):
    show(m)

## 4. Fake 포트 정의 — 외부 호출 없이 파이프라인 검증

세 포트(`LLMPort`, `ImageGeneratorPort`, `S3Port`) 프로토콜만 만족하면 된다.

In [ ]:
from uuid import uuid4


class FakeLLM:
    def __init__(self, caption: str = "오늘 방 청소 완료! 기분 최고 ✨") -> None:
        self.caption = caption
        self.calls: list[str] = []

    async def generate(self, prompt: str) -> str:
        self.calls.append(prompt)
        return self.caption


class FailingLLM:
    async def generate(self, prompt: str) -> str:
        raise CaptionGenerationError("LLM 서버 오류")


class FakeImageGenerator:
    def __init__(self, image_bytes: bytes = b"fake_image_bytes") -> None:
        self.image_bytes = image_bytes
        self.calls: list[tuple[str, str]] = []

    async def generate_img2img(self, reference_url: str, prompt: str) -> bytes:
        self.calls.append((reference_url, prompt))
        return self.image_bytes


class FailingImageGenerator:
    async def generate_img2img(self, reference_url: str, prompt: str) -> bytes:
        raise ImageGenerationError("이미지 생성 서버 오류")


class FakeS3:
    def __init__(self, url: str = "https://s3.example.com/feeds/result.png") -> None:
        self.url = url
        self.calls: list[tuple[str, bytes]] = []

    async def upload(self, key: str, data: bytes) -> str:
        self.calls.append((key, data))
        return self.url


class FailingS3:
    async def upload(self, key: str, data: bytes) -> str:
        raise S3UploadError("S3 연결 오류")


def make_input(**overrides) -> FeedGenerationInput:
    data = dict(
        quest=QuestRef(quest_id=uuid4(), quest_text="방 청소하기"),
        character=CharacterRef(
            character_id=uuid4(),
            name="몽글이",
            personality="밝고 활발함",
            speech_style="반말, 이모티콘 자주 사용",
            appearance_keywords=["분홍색 머리", "큰 눈", "귀여운"],
            image_url="https://s3.example.com/characters/test.png",
        ),
    )
    data.update(overrides)
    return FeedGenerationInput(**data)


def make_ports(**overrides) -> Ports:
    defaults = dict(llm=FakeLLM(), image_generator=FakeImageGenerator(), s3=FakeS3())
    defaults.update(overrides)
    return Ports(**defaults)


print("helper 준비 완료")

## 5. 시나리오 A — 정상 흐름

기대: `GeneratedFeed` 반환, `image_url` https://, `caption` ≤140자 + 한글 포함.

In [ ]:
img_gen = FakeImageGenerator()
s3 = FakeS3()
llm = FakeLLM()
inp = make_input()

result = await pipeline.run(inp, ports=Ports(llm=llm, image_generator=img_gen, s3=s3))

print(f"type        : {type(result).__name__}")
print(f"character_id: {result.character_id}  ==  input? {result.character_id == inp.character.character_id}")
print(f"quest_id    : {result.quest_id}  ==  input? {result.quest_id == inp.quest.quest_id}")
print(f"image_url   : {result.image_url}")
print(f"caption     : {result.caption!r}")
print(f"caption len : {len(result.caption)} ≤ 140? {len(result.caption) <= 140}")
print()
print(f"img2img reference_url: {img_gen.calls[0][0]!r}")
print(f"S3 key               : {s3.calls[0][0]!r}")
print(f"LLM 프롬프트 첫줄    : {llm.calls[0].splitlines()[0]!r}")

## 6. 시나리오 B — Img2Img 실패 (RetryPolicy max=3)

`FailingImageGenerator` 는 항상 `ImageGenerationError` 를 던진다.
LangGraph `RetryPolicy(max_attempts=3)` 가 3회 재시도 후 예외를 전파해야 한다.

In [ ]:
try:
    await pipeline.run(make_input(), ports=make_ports(image_generator=FailingImageGenerator()))
    print("ERROR: 예외가 발생하지 않았습니다")
except ImageGenerationError as e:
    print(f"✅ ImageGenerationError 전파됨: {e}")
except Exception as e:
    print(f"❌ 예상치 못한 예외: {type(e).__name__}: {e}")

## 7. 시나리오 C — S3 업로드 실패 (RetryPolicy max=3)

이미지 생성은 성공하지만 S3 업로드가 항상 실패.
`S3UploadError` 가 전파되어야 한다.

In [ ]:
try:
    await pipeline.run(make_input(), ports=make_ports(s3=FailingS3()))
    print("ERROR: 예외가 발생하지 않았습니다")
except S3UploadError as e:
    print(f"✅ S3UploadError 전파됨: {e}")
except Exception as e:
    print(f"❌ 예상치 못한 예외: {type(e).__name__}: {e}")

## 8. 시나리오 D — 캡션 한국어 미포함 (C1 위반)

LLM 이 영어 캡션을 반환하면 `validate_caption` 이 `CaptionValidationError(code='no_korean')` 를 던진다.
재시도 없음 — 프롬프트 문제이므로 호출자에게 즉시 전파.

In [ ]:
try:
    await pipeline.run(
        make_input(),
        ports=make_ports(llm=FakeLLM(caption="Cleaned my room! Great day!")),
    )
    print("ERROR: 예외가 발생하지 않았습니다")
except CaptionValidationError as e:
    print(f"✅ CaptionValidationError 전파됨")
    print(f"   code   : {e.code!r}")
    print(f"   message: {e.message!r}")

## 9. 시나리오 E — 캡션 140자 초과 (C2 위반)

LLM 이 141자 캡션을 반환하면 `CaptionValidationError(code='caption_too_long')` 가 발생해야 한다.

In [ ]:
long_caption = "가" * 141
print(f"테스트 캡션 길이: {len(long_caption)}자")

try:
    await pipeline.run(
        make_input(),
        ports=make_ports(llm=FakeLLM(caption=long_caption)),
    )
    print("ERROR: 예외가 발생하지 않았습니다")
except CaptionValidationError as e:
    print(f"✅ CaptionValidationError 전파됨")
    print(f"   code   : {e.code!r}")
    print(f"   message: {e.message!r}")

## 10. 시나리오 F — 입력 검증 실패 (image_url 공백)

`validate` 노드가 `character.image_url` 이 공백이면 `InputValidationError` 를 던진다.

In [ ]:
blank_url_input = make_input(
    character=CharacterRef(
        character_id=uuid4(),
        name="몽글이",
        personality="밝음",
        speech_style="반말",
        appearance_keywords=["귀여운"],
        image_url="   ",
    )
)

try:
    await pipeline.run(blank_url_input, ports=make_ports())
    print("ERROR: 예외가 발생하지 않았습니다")
except InputValidationError as e:
    print(f"✅ InputValidationError 전파됨")
    print(f"   code   : {e.code!r}")
    print(f"   message: {e.message!r}")

## 11. (선택) 실제 Mi:dm 어댑터로 캡션 실주행

`adapters/feed_generation/midm_llm.py` 를 사용한 실주행.
이미지 생성·S3 는 여전히 Fake 사용.

환경변수 `MIDM_BASE_URL`, `MIDM_MODEL` 이 설정돼 있어야 한다.

In [ ]:
import os

if not os.getenv("MIDM_BASE_URL"):
    print("MIDM_BASE_URL 미설정 → 실주행 스킵.")
else:
    try:
        from adapters.feed_generation.midm_llm import MidmLLM

        real_result = await pipeline.run(
            make_input(),
            ports=make_ports(
                llm=MidmLLM(
                    model=os.environ["MIDM_MODEL"],
                    base_url=os.environ["MIDM_BASE_URL"],
                )
            ),
        )
        print(f"caption ({len(real_result.caption)}자): {real_result.caption}")
        print(f"image_url: {real_result.image_url}")
    except Exception as exc:
        print(f"실주행 실패: {exc!r}")